# 电话叫车问题 (DARP)

**类别:** 路径

来源: [https://www.hexaly.com/templates/dial-a-ride-problem-darp](https://www.hexaly.com/templates/dial-a-ride-problem-darp)


## 问题描述

**在电话叫车问题 (DARP)** 中,一组车辆必须将客户从一个地点运送到另一个地点。车辆从一个共同的配送中心出发并最终返回配送中心,且具有最大载客能力。客户必须被恰好一辆车接送。客户在其接送地点均需要装车时间,且其接送时间必须落在给定的时间窗口内。每个客户接送时间之间的延迟不得超过某个上限。目标是为每辆卡车分配一个客户序列,同时最小化总延迟和总行驶距离。

	

### 学到的要点

- 添加 [列表决策变量](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html) 来建模卡车的客户序列
- 使用一个 [递归 lambda 函数](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html#special-case) 来定义数组,以计算客户的接送时间
- 添加 [多个目标](https://www.hexaly.com/docs/last/modelingprinciples/multiobjectiveresolution.html),并将延迟建模为 [软约束](https://www.hexaly.com/docs/last/modelingprinciples/modelingprinciples.html#distinguish-constraints-from-first-priority-objectives)


## 数据

我们提供来自 [Chassaing 等人数据集](https://perso.isima.fr/~lacomme/Maxime/Real_life_instances/Real_life_instances.php) 的电话叫车问题 (DARP) 实例。每个数据文件都是一个 JSON 文件,包含:

- 车辆数量
- 节点数量
- 最大行驶时间(对所有客户相同)
- 车辆载客能力
- 用于计算两个不同节点之间行驶时间的缩放因子
- 车辆的速度
- 客户数量
- 每对节点之间的距离矩阵
- 关于配送中心的信息:

- 标识符(节点索引)
- 装车时间
- 配送中心处的客户数量
- 路线的开始时间
- 路线的最迟结束时间
- 在配送中心接送的客户的最大行驶时间(若有)
- 关于每位客户的信息。对每个节点,文件给出:

- 请求车辆的客户数量
- 关于接送的信息:接送节点的索引、最早接送时间、最晚接送时间、车辆内的装车时间、最大行驶时间
- 关于送达的信息:送达节点的索引、最早送达时间、最晚送达时间、离开车辆所需的装车时间、最大行驶时间


## 模型

电话叫车问题 (DARP) 的 Hexaly 模型使用列表决策变量来建模每辆卡车访问的节点(客户接送点)序列,以及浮点型决策变量来表示在配送中心和每个节点的等待时间。我们对列表使用 **partition** 约束以确保每个节点被访问恰好一次。

每辆车的载客量在路线过程中会变化:在接送时增加,在送达时减少。我们使用 [**递归数组**](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html#special-case) 来计算车辆随时间的载客量:访问一个节点之后的载客量等于访问前一个节点之后的载客量加上当前节点的载客量。然后我们使用可变参 `and` 算子来确保在路线的任何时刻载客量都不超过容量上限。

由于时间窗口可能难以严格满足,它们被视为第一优先级目标而不是硬约束。如果车辆早到,则必须等待该节点的开放时间。如果迟到,我们会测量并惩罚该延迟。我们使用 [**递归数组**](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html#special-case) 来计算每辆车对每个节点访问的结束时间。对于每辆车和每个节点,结束时间为以下各项之和:

- 节点等待时间
- 节点装车时间
- 以下两者的最大值:

- 上一次访问的结束时间加上行驶时间(对于第一次访问,即为从配送中心出发的行驶时间)
- 该节点允许的最早到达时间

利用 **find** 和 [**indexOf**](https://www.hexaly.com/docs/last/modelerreference/standardlibrary/builtinfunctions.html#indexOf) 算子,我们确保每位客户由同一辆车接送。

最后,我们按字典序最小化以下各项:

- 总路线延迟(包括返回配送中心的延迟)
- 总客户延迟
- 总行驶距离


## Python 实现


In [ ]:
# Copyright (c) Hexaly. Permission is hereby granted to use, copy,
# and modify this code for applications developed with Hexaly.
import hexaly.optimizer
import sys
import json

def read_data(filename):
    with open(filename) as f:
        return json.load(f)
    
def read_input_darp(instance_file):
    instance = read_data(instance_file)

    nb_clients = instance["nbClients"]
    nb_nodes = instance["nbNodes"]
    nb_vehicles = instance["nbVehicles"]
    depot_tw_end = instance["depot"]["twEnd"]
    capacity = instance["capacity"]
    scale = instance["scale"]

    quantities = [-1 for i in range(2 * nb_clients)]
    distances = instance["distanceMatrix"]
    starts = [-1.0 for i in range(2 * nb_clients)]
    ends = [-1.0 for i in range(2 * nb_clients)]
    loading_times = [-1.0 for i in range(2 * nb_clients)]
    max_travel_times = [-1.0 for i in range(2 * nb_clients)]
    for k in range(nb_clients):
        quantities[k] = instance["clients"][k]["nbClients"]
        quantities[k+nb_clients] = -instance["clients"][k]["nbClients"]

        starts[k] = instance["clients"][k]["pickup"]["start"]
        ends[k] = instance["clients"][k]["pickup"]["end"]
        
        starts[k+nb_clients] = instance["clients"][k]["delivery"]["start"]
        ends[k+nb_clients] = instance["clients"][k]["delivery"]["end"]

        loading_times[k] = instance["clients"][k]["pickup"]["loadingTime"]
        loading_times[k+nb_clients] = instance["clients"][k]["delivery"]["loadingTime"]

        max_travel_times[k] = instance["clients"][k]["pickup"]["maxTravelTime"]
        max_travel_times[k+nb_clients] = instance["clients"][k]["delivery"]["maxTravelTime"]
        
    factor = 1.0 / (scale * instance["speed"])

    distance_warehouse = [-1.0 for i in range(nb_nodes)]
    time_warehouse = [-1.0 for i in range(nb_nodes)]
    distance_matrix = [[-1.0 for i in range(nb_nodes)] for j in range(nb_nodes)]
    time_matrix = [[-1.0 for i in range(nb_nodes)] for j in range(nb_nodes)]
    for i in range(nb_nodes):
        distance_warehouse[i] = distances[0][i+1]
        time_warehouse[i] = distance_warehouse[i] * factor
        for j in range(nb_nodes):
            distance_matrix[i][j] = distances[i+1][j+1]
            time_matrix[i][j] = distance_matrix[i][j] * factor

    return nb_clients, nb_nodes, nb_vehicles, depot_tw_end, capacity, scale, quantities, \
        starts, ends, loading_times, max_travel_times, distance_warehouse, time_warehouse, \
        distance_matrix, time_matrix

def main(instance_file, str_time_limit, sol_file):

    nb_clients, nb_nodes, nb_vehicles, depot_tw_end, capacity, scale, quantities_data, \
        starts_data, ends_data, loading_times_data, max_travel_times, distance_warehouse_data, \
        time_warehouse_data, distance_matrix_data, time_matrix_data = read_input_darp(instance_file)

    with hexaly.optimizer.HexalyOptimizer() as optimizer:
        model = optimizer.model

        # routes[k] represents the nodes visited by vehicle k
        routes = [model.list(nb_nodes) for k in range(nb_vehicles)]
        depot_starts = [model.float(0, depot_tw_end) for k in range(nb_vehicles)]
        # Each node is taken by one vehicle
        model.constraint(model.partition(routes))

        quantities = model.array(quantities_data)
        time_warehouse = model.array(time_warehouse_data)
        time_matrix = model.array(time_matrix_data)
        loading_times = model.array(loading_times_data)
        starts = model.array(starts_data)
        ends = model.array(ends_data)
        # waiting[k] is the waiting time at node k
        waiting = [model.float(0, depot_tw_end) for k in range(nb_nodes)]
        waiting_array = model.array(waiting)
        distance_matrix = model.array(distance_matrix_data)
        distance_warehouse = model.array(distance_warehouse_data)

        times = [None] * nb_vehicles
        lateness = [None] * nb_vehicles
        home_lateness = [None] * nb_vehicles
        route_distances = [None] * nb_vehicles

        for k in range(nb_vehicles):
            route = routes[k]
            c = model.count(route)

            demand_lambda = model.lambda_function(lambda i, prev: prev + quantities[route[i]])
            # route_quantities[k][i] indicates the number of clients in vehicle k
            # at its i-th taken node
            route_quantities = model.array(model.range(0, c), demand_lambda)
            quantity_lambda = model.lambda_function(lambda i: route_quantities[i] <= capacity)
            # Vehicles have a maximum capacity
            model.constraint(model.and_(model.range(0, c), quantity_lambda))

            times_lambda = model.lambda_function(
                lambda i, prev: model.max(
                    starts[route[i]],
                    model.iif(
                        i == 0,
                        depot_starts[k] + time_warehouse[route[0]],
                        prev + time_matrix[route[i-1]][route[i]]
                    )
                ) + waiting_array[route[i]] + loading_times[route[i]]
            )
            # times[k][i] is the time at which vehicle k leaves the i-th node
            # (after waiting and loading time at node i)
            times[k] = model.array(model.range(0, c), times_lambda)

            lateness_lambda = model.lambda_function(
                lambda i: model.max(
                    0,
                    times[k][i] - loading_times[route[i]] - ends[route[i]]
                )
            )
            # Total lateness of the k-th route
            lateness[k] = model.sum(model.range(0, c), lateness_lambda)

            home_lateness[k] = model.iif(
                c > 0,
                model.max(0, times[k][c-1] + time_warehouse[route[c-1]] - depot_tw_end),
                0
            )

            route_dist_lambda = model.lambda_function(
                lambda i: distance_matrix[route[i-1]][route[i]]
            )
            route_distances[k] = model.sum(
                model.range(1, c),
                route_dist_lambda
            ) + model.iif(
                c > 0,
                distance_warehouse[route[0]] + distance_warehouse[route[c-1]],
                0
            )

        routes_array = model.array(routes)
        times_array = model.array(times)
        client_lateness = [None] * nb_clients
        for k in range(nb_clients):
            # For each pickup node k, its associated delivery node is k + nb_clients
            pickup_list_index = model.find(routes_array, k)
            delivery_list_index = model.find(routes_array, k + nb_clients)
            # A client picked up in route i is delivered in route i
            model.constraint(pickup_list_index == delivery_list_index)

            client_list = routes_array[pickup_list_index]
            pickup_index = model.index(client_list, k)
            delivery_list = routes_array[delivery_list_index]
            delivery_index = model.index(delivery_list, k + nb_clients)
            # Pickup before delivery
            model.constraint(pickup_index < delivery_index)

            pickup_time = times_array[pickup_list_index][pickup_index]
            delivery_time = times_array[delivery_list_index][delivery_index] \
                - loading_times[k + nb_clients]
            travel_time = delivery_time - pickup_time
            client_lateness[k] = model.max(travel_time - max_travel_times[k], 0)

        total_lateness = model.sum(lateness + home_lateness)
        total_client_lateness = model.sum(client_lateness)
        total_distance = model.sum(route_distances)

        model.minimize(total_lateness)
        model.minimize(total_client_lateness)
        model.minimize(total_distance / scale)

        model.close()
        optimizer.param.time_limit = int(str_time_limit)
        optimizer.solve()

        #
        # Write the solution in a file with the following format:
        #  - total lateness on the routes, total client lateness, and total distance
        #  - for each vehicle, the depot start time, the nodes visited (omitting the start/end at the
        # depot), and the waiting time at each node
        #
        if sol_file is not None:
            with open(sol_file, 'w') as f:
                f.write("%d %d %.2f\n" % (
                    total_lateness.value,
                    total_client_lateness.value,
                    total_distance.value
                ))
                for k in range(nb_vehicles):
                    f.write("Vehicle %d (%.2f): " %(k + 1, depot_starts[k].value))
                    for node in routes[k].value:
                        f.write("%d (%.2f), " % (node, waiting[node].value))
                    f.write("\n")
    return 0

if __name__ == '__main__':
    if len(sys.argv) < 2:
        print("Usage: python darp.py input_file [output_file] [time_limit]")
        sys.exit(1)

    instance_file = sys.argv[1]
    sol_file = sys.argv[2] if len(sys.argv) > 2 else None
    str_time_limit = sys.argv[3] if len(sys.argv) > 3 else "20"

    main(instance_file, str_time_limit, sol_file)
